# Ridge, Lasso and Elastic Net Regression
Works on **any dataset** (CSV / Excel). Run the cells top to bottom. A file explorer window will pop up to pick your dataset.

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import (LinearRegression, Ridge, Lasso, ElasticNet,
                                  RidgeCV, LassoCV, ElasticNetCV, lasso_path)
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

## 2. Select Dataset from File Explorer

In [ ]:
import tkinter as tk
from tkinter import filedialog

root = tk.Tk()
root.withdraw()
root.attributes('-topmost', True)   # bring dialog to front
file_path = filedialog.askopenfilename(
    title="Select your dataset",
    filetypes=[("Data files", "*.csv *.xlsx *.xls *.txt *.tsv"), ("All files", "*.*")]
)
root.destroy()

if not file_path:
    raise SystemExit("No file selected. Re-run this cell.")

print("Selected file:", file_path)

if file_path.endswith('.csv'):
    df = pd.read_csv(file_path)
elif file_path.endswith(('.xlsx', '.xls')):
    df = pd.read_excel(file_path)
else:
    df = pd.read_csv(file_path, sep=None, engine='python')   # txt / tsv auto-detect separator

print("Shape:", df.shape)
df.head()

## 3. Explore the Data

In [ ]:
print(df.info())
print("\nMissing values:\n", df.isnull().sum())
df.describe()

## 4. Choose Target Column

In [ ]:
print("Columns:")
for i, c in enumerate(df.columns):
    print(f"{i}: {c}")

target = input("\nEnter the TARGET column name (or index number): ").strip()
if target.isdigit() and target not in df.columns:
    target = df.columns[int(target)]
print("Target selected:", target)

## 5. Preprocessing (missing values, categorical encoding)

In [ ]:
data = df.copy()

# drop rows where target is missing
data = data.dropna(subset=[target])

# target must be numeric for regression
data[target] = pd.to_numeric(data[target], errors='coerce')
data = data.dropna(subset=[target])

y = data[target]
X = data.drop(columns=[target])

# fill missing values: numeric -> median, categorical -> mode
for col in X.columns:
    if X[col].dtype == 'object' or str(X[col].dtype) == 'category':
        X[col] = X[col].fillna(X[col].mode()[0])
    else:
        X[col] = X[col].fillna(X[col].median())

# one-hot encode categorical columns
X = pd.get_dummies(X, drop_first=True)
X = X.astype(float)

print("Features shape:", X.shape)
X.head()

## 6. Correlation Heatmap (optional, skipped if too many features)

In [ ]:
if X.shape[1] <= 25:
    plt.figure(figsize=(10, 8))
    sns.heatmap(pd.concat([X, y], axis=1).corr(), annot=True, fmt='.2f', cmap='coolwarm')
    plt.title("Correlation Matrix")
    plt.show()
else:
    print("Too many features for heatmap - skipped.")

## 7. Train-Test Split and Standardization
Regularization is sensitive to feature scale, so scaling is **mandatory**.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print("Train:", X_train_s.shape, " Test:", X_test_s.shape)

## 8. Baseline: Ordinary Least Squares

In [ ]:
def evaluate(name, model, Xtr, Xte, ytr, yte):
    ytr_pred = model.predict(Xtr)
    yte_pred = model.predict(Xte)
    return {
        'Model': name,
        'Train R2': r2_score(ytr, ytr_pred),
        'Test R2': r2_score(yte, yte_pred),
        'Test MSE': mean_squared_error(yte, yte_pred),
        'Test RMSE': np.sqrt(mean_squared_error(yte, yte_pred)),
        'Test MAE': mean_absolute_error(yte, yte_pred)
    }

ols = LinearRegression().fit(X_train_s, y_train)
results = [evaluate('Linear Regression', ols, X_train_s, X_test_s, y_train, y_test)]
pd.DataFrame(results)

## 9. Ridge Regression (L2 penalty)
Minimizes: `RSS + alpha * sum(beta^2)` - shrinks coefficients, never exactly zero.

In [ ]:
alphas = np.logspace(-3, 3, 100)

ridge_cv = RidgeCV(alphas=alphas, cv=5).fit(X_train_s, y_train)
print("Best alpha (Ridge):", ridge_cv.alpha_)

ridge = Ridge(alpha=ridge_cv.alpha_).fit(X_train_s, y_train)
results.append(evaluate('Ridge', ridge, X_train_s, X_test_s, y_train, y_test))
pd.DataFrame(results)

## 10. Lasso Regression (L1 penalty)
Minimizes: `RSS + alpha * sum(|beta|)` - shrinks some coefficients to exactly zero (feature selection).

In [ ]:
lasso_cv = LassoCV(alphas=None, cv=5, random_state=42, max_iter=10000).fit(X_train_s, y_train)
print("Best alpha (Lasso):", lasso_cv.alpha_)

lasso = Lasso(alpha=lasso_cv.alpha_, max_iter=10000).fit(X_train_s, y_train)
results.append(evaluate('Lasso', lasso, X_train_s, X_test_s, y_train, y_test))
print("Features kept:", np.sum(lasso.coef_ != 0), "of", X.shape[1])
pd.DataFrame(results)

## 11. Elastic Net (L1 + L2 penalty)
Minimizes: `RSS + alpha * [ l1_ratio * sum(|beta|) + (1 - l1_ratio)/2 * sum(beta^2) ]`

In [ ]:
enet_cv = ElasticNetCV(l1_ratio=[.1, .3, .5, .7, .9, .95, 1], cv=5,
                       random_state=42, max_iter=10000).fit(X_train_s, y_train)
print("Best alpha   (ElasticNet):", enet_cv.alpha_)
print("Best l1_ratio(ElasticNet):", enet_cv.l1_ratio_)

enet = ElasticNet(alpha=enet_cv.alpha_, l1_ratio=enet_cv.l1_ratio_, max_iter=10000).fit(X_train_s, y_train)
results.append(evaluate('Elastic Net', enet, X_train_s, X_test_s, y_train, y_test))
print("Features kept:", np.sum(enet.coef_ != 0), "of", X.shape[1])
pd.DataFrame(results)

## 12. Model Comparison

In [ ]:
res_df = pd.DataFrame(results).set_index('Model')
display(res_df.round(4))

res_df[['Train R2', 'Test R2']].plot(kind='bar', figsize=(8, 5))
plt.title("R2 Comparison"); plt.ylabel("R2"); plt.xticks(rotation=0); plt.grid(alpha=.3)
plt.show()

best = res_df['Test RMSE'].idxmin()
print("Best model (lowest Test RMSE):", best)

## 13. Coefficient Comparison

In [ ]:
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'OLS': ols.coef_,
    'Ridge': ridge.coef_,
    'Lasso': lasso.coef_,
    'ElasticNet': enet.coef_
}).set_index('Feature')

display(coef_df.round(4))

coef_df.plot(kind='bar', figsize=(14, 6))
plt.title("Coefficients: OLS vs Ridge vs Lasso vs Elastic Net")
plt.axhline(0, color='black', lw=.8)
plt.ylabel("Coefficient"); plt.grid(alpha=.3)
plt.show()

print("Features removed by Lasso      :", list(coef_df.index[coef_df['Lasso'] == 0]))
print("Features removed by ElasticNet :", list(coef_df.index[coef_df['ElasticNet'] == 0]))

## 14. Coefficient Shrinkage Path (effect of alpha)

In [ ]:
# Ridge path
ridge_coefs = []
for a in alphas:
    ridge_coefs.append(Ridge(alpha=a).fit(X_train_s, y_train).coef_)

plt.figure(figsize=(13, 5))
plt.subplot(1, 2, 1)
plt.plot(alphas, ridge_coefs)
plt.xscale('log'); plt.xlabel('alpha'); plt.ylabel('Coefficients')
plt.title('Ridge Path'); plt.grid(alpha=.3)

# Lasso path
alphas_l, coefs_l, _ = lasso_path(X_train_s, y_train)
plt.subplot(1, 2, 2)
plt.plot(alphas_l, coefs_l.T)
plt.xscale('log'); plt.xlabel('alpha'); plt.ylabel('Coefficients')
plt.title('Lasso Path'); plt.grid(alpha=.3)
plt.show()

## 15. Cross-Validation Error vs Alpha

In [ ]:
plt.figure(figsize=(13, 4))

# Ridge
ridge_scores = [-cross_val_score(Ridge(alpha=a), X_train_s, y_train, cv=5,
                                 scoring='neg_mean_squared_error').mean() for a in alphas]
plt.subplot(1, 2, 1)
plt.plot(alphas, ridge_scores)
plt.axvline(ridge_cv.alpha_, color='r', ls='--', label=f'best={ridge_cv.alpha_:.3f}')
plt.xscale('log'); plt.xlabel('alpha'); plt.ylabel('CV MSE'); plt.title('Ridge'); plt.legend()

# Lasso
plt.subplot(1, 2, 2)
plt.plot(lasso_cv.alphas_, lasso_cv.mse_path_.mean(axis=1))
plt.axvline(lasso_cv.alpha_, color='r', ls='--', label=f'best={lasso_cv.alpha_:.3f}')
plt.xscale('log'); plt.xlabel('alpha'); plt.ylabel('CV MSE'); plt.title('Lasso'); plt.legend()
plt.show()

## 16. Actual vs Predicted and Residuals

In [ ]:
models = {'Ridge': ridge, 'Lasso': lasso, 'Elastic Net': enet}

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for i, (name, m) in enumerate(models.items()):
    pred = m.predict(X_test_s)
    axes[0, i].scatter(y_test, pred, alpha=.6)
    axes[0, i].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
    axes[0, i].set_title(f'{name}: Actual vs Predicted')
    axes[0, i].set_xlabel('Actual'); axes[0, i].set_ylabel('Predicted')

    axes[1, i].scatter(pred, y_test - pred, alpha=.6)
    axes[1, i].axhline(0, color='r', ls='--')
    axes[1, i].set_title(f'{name}: Residuals')
    axes[1, i].set_xlabel('Predicted'); axes[1, i].set_ylabel('Residual')
plt.tight_layout()
plt.show()

## 17. Predict on New Data (optional)

In [ ]:
# Example: predict using first 5 rows of test set
sample = X_test.iloc[:5]
sample_s = scaler.transform(sample)

pred_df = pd.DataFrame({
    'Actual': y_test.iloc[:5].values,
    'Ridge': ridge.predict(sample_s),
    'Lasso': lasso.predict(sample_s),
    'ElasticNet': enet.predict(sample_s)
})
pred_df.round(3)

## Quick Theory (for viva)
| | Ridge | Lasso | Elastic Net |
|---|---|---|---|
| Penalty | L2: `alpha * sum(b^2)` | L1: `alpha * sum(abs(b))` | Mix of L1 and L2 |
| Coefficients | Shrunk toward 0, never exactly 0 | Some become exactly 0 | Some become exactly 0 |
| Feature selection | No | Yes | Yes |
| Multicollinearity | Handles well (shrinks correlated features together) | Picks one of the correlated features arbitrarily | Handles well (grouping effect) |
| Best when | Many small/medium effects, p < n | Few important features, sparse model | Correlated features, p > n |
| Hyperparameters | alpha | alpha | alpha, l1_ratio |

- **alpha = 0** gives ordinary least squares; larger alpha means stronger shrinkage (more bias, less variance).
- Always **standardize** features before regularization.
- Choose alpha by **cross-validation**.